# 🖼️ Abhimanyu Rajput — Image Analyst
## Multimodal Crime / Incident Report Analyzer

**Data Type:** Crime scene / accident scene photographs  
**Objective:** Detect objects, classify scenes, extract text from images → structured CSV

### Pipeline:
1. Load scene images
2. Object detection with YOLOv8
3. Scene classification
4. OCR for visible text (license plates, signs)
5. Output structured CSV


In [1]:
# =============================================
# CELL 1: Install Dependencies
# =============================================
!pip install -q ultralytics opencv-python-headless pytesseract pillow pandas roboflow
!apt-get install -y -q tesseract-ocr > /dev/null 2>&1


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 42.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.8/95.8 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 66.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 94.0 MB/s eta 0:00:00


In [2]:
# =============================================
# CELL 2: Import Libraries
# =============================================
import cv2
import pytesseract
import pandas as pd
import numpy as np
import os, glob
from PIL import Image
from ultralytics import YOLO
import warnings
warnings.filterwarnings('ignore')

os.makedirs("outputs", exist_ok=True)

print("Loading YOLOv8 model...")
model = YOLO("yolov8n.pt")
print(f"Model loaded! {len(model.names)} classes")



Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Loading YOLOv8 model...
Model loaded! 80 classes


In [3]:
# =============================================
# CELL 3: Download Fire Detection Dataset
# =============================================
!pip install -q roboflow

from roboflow import Roboflow
import glob, os

# Use Colab secrets for API key (set in Colab: Runtime > Secrets)
from google.colab import userdata
api_key = userdata.get('ROBOFLOW_API_KEY')

rf = Roboflow(api_key=api_key)
project = rf.workspace("arkan-jrix2").project("fire-detection-my95a")
version = project.version(1)

# Remove old broken download
import shutil
if os.path.exists("/content/Fire-Detection-1"):
    shutil.rmtree("/content/Fire-Detection-1")

# Download fresh
dataset = version.download("yolov8", overwrite=True)

# Check what got downloaded
print(f"\nDataset location: {dataset.location}")
for root, dirs, files in os.walk(dataset.location):
    level = root.replace(dataset.location, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/  ({len(files)} files)")

# Find all images
all_images = []
for ext in ["*.jpg", "*.png", "*.jpeg"]:
    all_images += glob.glob(os.path.join(dataset.location, "**", ext), recursive=True)
all_images = sorted(all_images)

print(f"\nTotal images: {len(all_images)}")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Fire-Detection-1 in yolov8:: 100%|██████████| 45514/45514 [00:08<00:00, 5251.56it/s] 



Dataset location: /content/Fire-Detection-1
Fire-Detection-1/  (3 files)
  train/  (0 files)
    images/  (21633 files)
    labels/  (21633 files)
  test/  (0 files)
    images/  (380 files)
    labels/  (380 files)
  valid/  (0 files)
    images/  (738 files)
    labels/  (738 files)

Total images: 22751


In [4]:
# =============================================
# CELL 4: Object Detection with YOLOv8
# =============================================
# Object Detection with YOLOv8
def detect_objects(image_path, confidence_threshold=0.25):
    results = model(image_path, conf=confidence_threshold, verbose=False)
    detections = []
    for result in results:
        for box in result.boxes:
            cls_id = int(box.cls[0])
            cls_name = model.names[cls_id]
            conf = float(box.conf[0])
            bbox = box.xyxy[0].tolist()
            detections.append({
                "class": cls_name,
                "confidence": round(conf, 3),
                "bbox": [round(b, 1) for b in bbox]
            })
    return detections

# Run on dataset (limit for speed)
MAX_IMAGES = 5000
image_files = all_images[:MAX_IMAGES]

print(f"Running YOLOv8 on {len(image_files)} images...\n")
all_detections = {}
for i, filepath in enumerate(image_files, 1):
    filename = os.path.basename(filepath)
    detections = detect_objects(filepath)
    all_detections[filename] = {"detections": detections, "filepath": filepath}
    objects = [d["class"] for d in detections]
    if i <= 10 or detections:
        print(f"  [{i}/{len(image_files)}] {filename}: {objects if objects else 'no objects'}")

print(f"\nProcessed {len(image_files)} images.")

Running YOLOv8 on 5000 images...

  [1/5000] 0133_jpg.rf.43a82fa5d6c32dbc25002129cb72332d.jpg: no objects
  [2/5000] 0171_jpg.rf.0ca137dbb4563a0827f41f4dfe782956.jpg: ['chair', 'vase']
  [3/5000] 100_jpg.rf.55355f2e936333a94fefce2d9a7ada2d.jpg: ['bench']
  [4/5000] 114_png_jpg.rf.a6302409b3c52ec25a011dd264838e18.jpg: ['dining table', 'bottle', 'boat']
  [5/5000] 116_png_jpg.rf.a80a220a375f234f992b10e60b0b0b1b.jpg: ['bottle']
  [6/5000] 118_png_jpg.rf.c3c579e439d41ef21ef94b7a892db710.jpg: ['airplane', 'airplane']
  [7/5000] 1196_jpg.rf.0c3a0ac9812a31817c7ab3be63b557a3.jpg: ['car', 'car', 'car', 'car', 'car', 'car', 'person', 'person', 'car', 'car', 'car', 'person', 'car', 'car', 'bus', 'person', 'car', 'person']
  [8/5000] 119_png_jpg.rf.7445bd946299167e0d2c7e8142949dc3.jpg: ['airplane', 'bottle', 'bowl']
  [9/5000] 132_png_jpg.rf.a6af9be49c1bbe9ba7456823b6d8b476.jpg: no objects
  [10/5000] 135_png_jpg.rf.6b081b0707a1782239db01ba42713d33.jpg: no objects
  [13/5000] 151_png_jpg.rf.ed54e6

In [5]:
# =============================================
# CELL 5: Scene Classification
# =============================================

# Scene Classification
SCENE_RULES = {
    "Fire Scene": ["fire", "smoke", "flame"],
    "Vehicle Accident": ["car", "truck", "bus", "motorcycle"],
    "Assault / Violence": ["person", "knife"],
    "Theft / Break-in": ["person", "backpack", "handbag"],
}

def classify_scene(detections, filepath=None):
    detected_classes = set(d["class"] for d in detections)
    scene_scores = {}
    for scene_type, indicators in SCENE_RULES.items():
        score = sum(1 for ind in indicators if ind in detected_classes)
        if score > 0:
            scene_scores[scene_type] = score
    if scene_scores:
        return max(scene_scores, key=scene_scores.get)

    if filepath:
        img = cv2.imread(filepath)
        if img is not None:
            hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
            fire_mask = cv2.inRange(hsv, np.array([0,100,100]), np.array([30,255,255]))
            if np.sum(fire_mask > 0) / fire_mask.size > 0.05:
                return "Fire Scene"
    return "Unknown"

In [6]:
# =============================================
# CELL 6: OCR Text Extraction
# =============================================

# OCR Text Extraction
def extract_text_from_image(image_path):
    try:
        img = Image.open(image_path)
        text = pytesseract.image_to_string(img)
        text = " ".join(text.split())
        return text if len(text) > 3 else ""
    except:
        return ""

print("Extracting text via OCR...")
ocr_results = {}
for filename, data in all_detections.items():
    text = extract_text_from_image(data["filepath"])
    ocr_results[filename] = text
    if text:
        print(f"  {filename}: '{text[:60]}...'")

print("OCR complete.")


Extracting text via OCR...
  144_jpg.rf.4a65b14284a8c54829921349d0c9f33f.jpg: 'wee Piney eo We...'
  694_jpg.rf.adb3494132107c2742b9a3fc237559dc.jpg: 'baa ee 73 SEES aa Tee...'
  908_jpg.rf.4f44be6d1561b28c425680d4128e95c6.jpg: '— Ef alamy stock photo ee...'
  Img_1144_jpg.rf.3d70e92b18932e66da8e683df86ea6f6.jpg: '— UltimateChase.com...'
  Img_1150_jpg.rf.612e5e3b093e15b4ef0364f6a0490f95.jpg: '¥ UltimateChase.com...'
  Img_1165_jpg.rf.851020ef3beba812014c07aca4c4fd7d.jpg: 'UltimateChase.com...'
  Img_1175_jpg.rf.c3018c521d25a49d674d13d7d4f9790e.jpg: 'UltimateChase.com me eT a ap...'
  Img_1183_jpg.rf.648726fcf44ca88dcf7014fdf0f344b4.jpg: 'UltimateChase.com...'
  Img_1186_jpg.rf.65271eb9cff22e3838cb6dbbc75b8eed.jpg: 'UltimateChase.com...'
  Img_1196_jpg.rf.d9a2620159257a2be2d89692b6574ee2.jpg: 'UltimateChase.com esi Pe...'
  Img_1197_jpg.rf.cbc7c11f53bdd523011274bc816581d5.jpg: 'UltimateChase.com all 3 ible ry nea...'
  Img_1211_jpg.rf.17352a2c8f963ac98e48e55e07b18485.jpg: 'UltimateChas

In [7]:
# =============================================
# CELL 7: Generate Final Structured CSV
# =============================================
results = []
for idx, (filename, data) in enumerate(all_detections.items(), 1):
    detections = data["detections"]
    filepath = data["filepath"]

    objects_detected = ", ".join(set(d["class"] for d in detections)) if detections else "fire (color-based)"
    num_regions = len(detections)
    bboxes = f"{num_regions} regions detected" if num_regions > 0 else "color analysis"
    scene_type = classify_scene(detections, filepath)
    confidence = max((d["confidence"] for d in detections), default=0.0)

    if not detections:
        img = cv2.imread(filepath)
        if img is not None:
            hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
            fire_mask = cv2.inRange(hsv, np.array([0,100,100]), np.array([30,255,255]))
            confidence = round(min(np.sum(fire_mask > 0) / fire_mask.size * 10, 0.99), 2)

    text_extracted = ocr_results.get(filename, "")

    results.append({
        "Image_ID": f"IMG_{idx:03d}",
        "Scene_Type": scene_type,
        "Objects_Detected": objects_detected,
        "Bounding_Boxes": bboxes,
        "Text_Extracted": text_extracted,
        "Confidence_Score": confidence
    })

df_image_output = pd.DataFrame(results)
df_image_output.to_csv("outputs/image_analyst_output.csv", index=False)

print("=" * 70)
print("IMAGE ANALYST - FINAL OUTPUT")
print("=" * 70)
print(f"Total images processed: {len(df_image_output)}")
print(f"Output saved to: outputs/image_analyst_output.csv")
print("=" * 70)
df_image_output

IMAGE ANALYST - FINAL OUTPUT
Total images processed: 5000
Output saved to: outputs/image_analyst_output.csv


,Image_ID,Scene_Type,Objects_Detected,Bounding_Boxes,Text_Extracted,Confidence_Score
0,IMG_001,Unknown,fire (color-based),color analysis,,0.110
1,IMG_002,Fire Scene,"chair, vase",2 regions detected,,0.753
2,IMG_003,Unknown,bench,1 regions detected,,0.515
3,IMG_004,Unknown,"dining table, bottle, boat",3 regions detected,,0.412
4,IMG_005,Unknown,bottle,1 regions detected,,0.459
...,...,...,...,...,...,...
4995,IMG_4996,Fire Scene,fire (color-based),color analysis,UltimateChase.com,0.860
4996,IMG_4997,Fire Scene,fire (color-based),color analysis,UltimateChase.com,0.990
4997,IMG_4998,Fire Scene,fire (color-based),color analysis,UltimateChase.com,0.990
4998,IMG_4999,Fire Scene,fire (color-based),color analysis,,0.730


### ✅ Image Analyst Complete!
**Output:** `outputs/image_analyst_output.csv`

